In [1]:
import struct
import math
import numpy as np
import matplotlib.pyplot as plt

# entropy
def entropy(b, size=1):

    d = {}
    for i in range(0, len(b), size):
        block = b[i:i+size]
        d[block] = d.get(block, 0) + 1
    
    H = 0
    lenght = len(b) // size

    for i in d.values():
        if i > 0:
            p_i = i / lenght
            H -= p_i * math.log2(p_i)

    return H

# BWT
def bwt_encode_block(b):
    if not b:
        return b'', 0
    
    n = len(b)
    indices = list(range(n))
    indices.sort(key=lambda i: b[i:] + b[:i])
    
    orig_index = indices.index(0)
    last_column = bytes([b[(i - 1) % n] for i in indices])
    
    return last_column, orig_index

def bwt_decode_block(last_column, index):
    n = len(last_column)

    count = [0]*256
    for byte in last_column:
        count[byte] += 1
    
    total = 0
    for i in range(256):
        c = count[i]
        count[i] = total
        total += c
    
    next_pos = [0]*n
    current_count = [0]*256
    for i in range(n):
        byte = last_column[i]
        pos = count[byte] + current_count[byte]
        next_pos[i] = pos
        current_count[byte] += 1
    
    result = bytearray()
    row = index
    for i in range(n):
        byte = last_column[row]
        result.append(byte)
        row = next_pos[row]
    
    return bytes(result[::-1])

def bwt_encode(data, block_size=10000):
    if block_size is None:
        return bwt_encode_block(data)
    
    num_blocks = (len(data) + block_size - 1) // block_size
    output = struct.pack('<I', num_blocks)
    indices = []
    
    for i in range(0, len(data), block_size):
        chunk = data[i:i+block_size]
        col, idx = bwt_encode_block(chunk)
        indices.append(idx)
        output += struct.pack('<I', idx)
        output += struct.pack('<I', len(col))
        output += col
        
    return output, indices

def bwt_decode(data, index=None, blocked=False):
    if not blocked:
        return bwt_decode_block(data, index)
    
    offset = 0
    num_blocks = struct.unpack('<I', data[offset:offset+4])[0]
    offset += 4
    
    result = bytearray()
    for _ in range(num_blocks):
        idx = struct.unpack('<I', data[offset:offset+4])[0]
        length = struct.unpack('<I', data[offset+4:offset+8])[0]
        offset += 8
        
        block_data = data[offset:offset+length]
        offset += length
        
        result.extend(bwt_decode_block(block_data, idx))
        
    return bytes(result)

# MTF
def mtf_encode(b):
    alphabet = list(range(256))
    result = []

    for byte in b:
        index = alphabet.index(byte)
        result.append(index)
        alphabet.pop(index)
        alphabet.insert(0, byte)
    
    return bytes(result)

def mtf_decode(b):
    alphabet = list(range(256))
    result = []
    
    for index in b:
        byte = alphabet[index]
        result.append(byte)
        alphabet.pop(index)
        alphabet.insert(0, byte)
    
    return bytes(result)

# rle
def RLE(b, Ms=8, Mc=8):
    if not b:
        return b''
    
    symbol_size = Ms // 8
    control_size = Mc // 8
    max_count = (1 << (control_size * 8 - 1)) - 1  # резервируем бит флага
    
    result = bytearray()
    i = 0
    n = len(b)
    
    while i < n:
        # Считаем повторы текущего символа
        current = b[i:i+symbol_size]
        count = 1
        j = i + symbol_size
        while j + symbol_size <= n and b[j:j+symbol_size] == current and count < max_count:
            count += 1
            j += symbol_size
        
        if count >= 3:
            result.extend(count.to_bytes(control_size, 'big'))
            result.extend(current)
            i = j
        else:
            # Собираем литералы (неповторяющиеся символы)
            literal_start = i
            while i < n:
                sym = b[i:i+symbol_size]
                if len(sym) < symbol_size:
                    break
                # Проверяем, не начался ли повтор из 3+ символов
                next_sym = b[i+symbol_size:i+2*symbol_size] if i + 2*symbol_size <= n else None
                if next_sym and sym == next_sym:
                    # Проверяем длину потенциального повтора
                    rep_count = 1
                    k = i + symbol_size
                    while k + symbol_size <= n and b[k:k+symbol_size] == sym and rep_count < max_count:
                        rep_count += 1
                        k += symbol_size
                    if rep_count >= 3:
                        break  # начался выгодный повтор
                i += symbol_size
                if (i - literal_start) // symbol_size >= max_count:
                    break
            
            # Записываем литералы
            literal_bytes = b[literal_start:i]
            lit_count = len(literal_bytes) // symbol_size
            if lit_count > 0:
                result.extend(((lit_count | 0x80) if control_size == 1 else lit_count).to_bytes(control_size, 'big'))
                result.extend(literal_bytes)
    
    return bytes(result)

def deRLE(b, Ms=8, Mc=8):
    if not b:
        return b''
    
    symbol_size = Ms // 8
    control_size = Mc // 8
    result = bytearray()
    i = 0
    
    while i < len(b):
        control = int.from_bytes(b[i:i+control_size], 'big')
        i += control_size
        
        if control & 0x80:  # литерал
            count = control & 0x7F
            end = i + count * symbol_size
            result.extend(b[i:end])
            i = end
        else:  # повтор
            symbol = b[i:i+symbol_size]
            i += symbol_size
            for _ in range(control):
                result.extend(symbol)
    
    return bytes(result)

def read_rle_file(filename):
    with open(filename, 'rb') as f:
        meta = f.read(2)
        Ms, Mc = meta[0], meta[1]
        data = f.read()
    return Ms, Mc, data

def write_rle_file(filename, data, Ms, Mc):
    with open(filename, 'wb') as f:
        f.write(bytes([Ms, Mc]))
        f.write(data)

def compress_file_rle(input_path, output_path, Ms=16, Mc=16):
    with open(input_path, 'rb') as f:
        data = f.read()
    
    compressed = RLE(data, Ms, Mc)
    write_rle_file(output_path, compressed, Ms, Mc)

def decompress_file_rle(input_path, output_path):
    Ms, Mc, compressed = read_rle_file(input_path)
    
    decompressed = deRLE(compressed, Ms, Mc)
    
    with open(output_path, 'wb') as f:
        f.write(decompressed)

import os
def get_code_lengths(node, length=0, lengths=None):
    # Обход дерева для получения длин кодов для каждого символа
    if lengths is None:
        lengths = {}
    
    if len(node) == 2:  # Лист (символ, частота)
        # Обработка случая одного символа
        lengths[node[0]] = max(1, length)
        return lengths

    if len(node) == 4:  # Узел (None, вес, лево, право)
        get_code_lengths(node[2], length + 1, lengths)
        get_code_lengths(node[3], length + 1, lengths)
    
    return lengths

def generate_canonical_codes(lengths):
    # Сортируем символы: сначала по длине кода, затем по значению символа
    sorted_symbols = sorted(lengths.keys(), key=lambda s: (lengths[s], s))
    
    codes = {}
    code = 0
    prev_len = 0
    
    for symbol in sorted_symbols:
        curr_len = lengths[symbol]
        # Сдвигаем код на разницу длин
        code <<= (curr_len - prev_len)
        
        # Формируем битовую строку
        codes[symbol] = format(code, f'0{curr_len}b')
        
        code += 1
        prev_len = curr_len
        
    return codes

def frequency_model(b):
    d = {}
    for i in b:
        d[i] = d.get(i, 0) + 1
    return d

def huffman_encode(data, model):
    # Сортируем очередь по частоте
    queue = sorted([[char, freq] for char, freq in model.items()], key=lambda x: x[1])
    
    # Строим дерево
    while len(queue) > 1:
        one = queue.pop(0)
        two = queue.pop(0)
        # Узел: [None, вес, левый_потомок, правый_потомок]
        parent = [None, one[1] + two[1], one, two]
        
        # Вставляем обратно, сохраняя сортировку по весу
        insert_position = 0
        for i, item in enumerate(queue):
            if parent[1] > item[1]:
                insert_position = i + 1
            else:
                break
        queue.insert(insert_position, parent)
    
    tree = queue[0] if queue else None
    
    # 1. Получаем длины кодов из дерева
    lengths = get_code_lengths(tree) if tree else {}
    
    # 2. Генерируем канонические коды по длинам
    codes = generate_canonical_codes(lengths)
    
    # 3. Кодируем данные
    encoded = ''.join(codes[char] for char in data)
    
    return codes, encoded, lengths

def huffman_decode(encoded, lengths):
    if not lengths or not encoded:
        return b""
    
    # Восстанавливаем канонические коды для декодирования
    codes = generate_canonical_codes(lengths)
    reverse_codes = {v: k for k, v in codes.items()}
    
    result = []
    buffer = ""
    
    # Оптимизация: максимальная длина кода для ограничения поиска
    max_len = max(lengths.values()) if lengths else 0
    
    for bit in encoded:
        buffer += bit
        # Проверяем, есть ли такой код в таблице
        if len(buffer) <= max_len and buffer in reverse_codes:
            result.append(reverse_codes[buffer])
            buffer = ""
    
    return bytes(result)

def bits_to_bytes(bit_string):
    if not bit_string:
        return b"", 0
        
    padding = (8 - len(bit_string) % 8) % 8
    bit_string += '0' * padding
    
    result = []
    for i in range(0, len(bit_string), 8):
        byte_chunk = bit_string[i:i+8]
        result.append(int(byte_chunk, 2))
    
    return bytes(result), padding

def bytes_to_bits(data, total_bits):
    bit_string = ''.join(f'{byte:08b}' for byte in data)
    return bit_string[:total_bits]

def write_huffman_file(filename, codes, encoded_bits, lengths):
    compressed_bytes, padding = bits_to_bytes(encoded_bits)
    unique_symbols = len(codes)
    
    with open(filename, 'wb') as f:
        # 1. Количество уникальных символов
        f.write(bytes([unique_symbols]))

        # 2. Таблица: Символ + Длина кода
        for byte_val in sorted(codes.keys()):
            f.write(bytes([byte_val]))
            f.write(bytes([lengths[byte_val]]))
        
        # 3. Служебная информация
        f.write(bytes([padding]))
        f.write(len(encoded_bits).to_bytes(4, 'big'))
        
        # 4. Сжатые данные
        f.write(compressed_bytes)
    
def read_huffman_file(filename):
    with open(filename, 'rb') as f:
        # 1. Количество уникальных символов
        num_symbols = f.read(1)[0]
        
        # 2. Читаем длины кодов
        lengths = {}
        for _ in range(num_symbols):
            byte_val = f.read(1)[0]
            code_len = f.read(1)[0]
            lengths[byte_val] = code_len
        
        # 3. Служебная информация
        padding = f.read(1)[0]
        total_bits = int.from_bytes(f.read(4), 'big')
        
        # 4. Сжатые данные
        compressed_data = f.read()
        encoded_bits = bytes_to_bits(compressed_data, total_bits)
    
    return lengths, encoded_bits

def compress_file(input_path, output_path):
    with open(input_path, 'rb') as f:
        data = f.read()
    
    if not data:
        with open(output_path, 'wb') as f:
            f.write(bytes([0])) # 0 символов
        return

    model = frequency_model(data)
    codes, encoded_bits, lengths = huffman_encode(data, model)
    write_huffman_file(output_path, codes, encoded_bits, lengths)

def decompress_file(input_path, output_path):
    lengths, encoded_bits = read_huffman_file(input_path)
    
    if not lengths:
        with open(output_path, 'wb') as f:
            pass
        return

    result = huffman_decode(encoded_bits, lengths)
    
    with open(output_path, 'wb') as f:
        f.write(result)


def compress(input_path, output_path, bwt_block_size=2000, rle_ms=8, rle_mc=8):
    with open(input_path, 'rb') as f:
        original_data = f.read()
    
    original_size = len(original_data)

    if bwt_block_size is not None:
        bwt_data, indices = bwt_encode(original_data, bwt_block_size)
        bwt_blocked = True
    else:
        bwt_data, idx = bwt_encode_block(original_data)
        indices = [idx]
        bwt_blocked = False

    mtf_data = mtf_encode(bwt_data)

    rle_data = RLE(mtf_data, rle_ms, rle_mc)

    model = frequency_model(rle_data)
    codes, encoded_bits, lengths = huffman_encode(rle_data, model)
    
    # Сохраняем результат с метаданными
    with open(output_path, 'wb') as f:
        # Заголовок: [флаги] [размер_блока] [ms] [mc]
        
        # Флаги
        flags = 0
        if bwt_blocked:
            flags |= 1  # бит 0: блокированный BWT
        f.write(bytes([flags]))
        
        # Параметры BWT
        if bwt_blocked:
            f.write(struct.pack('<I', bwt_block_size))
            f.write(struct.pack('<I', len(indices)))
            for idx in indices:
                f.write(struct.pack('<I', idx))
        else:
            f.write(struct.pack('<I', 0))  # размер блока 0 означает безблочный
            f.write(struct.pack('<I', indices[0]))
        
        # Параметры RLE
        f.write(bytes([rle_ms]))
        f.write(bytes([rle_mc]))
        
        # Данные Huffman
        compressed_bytes, padding = bits_to_bytes(encoded_bits)
        unique_symbols = len(codes)
        f.write(struct.pack('<I', unique_symbols))
        for byte_val in sorted(codes.keys()):
            f.write(bytes([byte_val]))
            f.write(bytes([lengths[byte_val]]))
        f.write(bytes([padding]))
        f.write(len(encoded_bits).to_bytes(4, 'big'))
        f.write(compressed_bytes)

def decompress(input_path, output_path):
    with open(input_path, 'rb') as f:
        flags = f.read(1)[0]
        
        # Читаем параметры BWT
        bwt_block_size = struct.unpack('<I', f.read(4))[0]
        indices_count = struct.unpack('<I', f.read(4))[0]
        
        indices = []
        for _ in range(indices_count):
            indices.append(struct.unpack('<I', f.read(4))[0])
        
        bwt_blocked = (flags & 1) != 0
        
        # Читаем параметры RLE
        rle_ms = f.read(1)[0]
        rle_mc = f.read(1)[0]
        
        # Читаем данные Huffman
        unique_symbols = struct.unpack('<I', f.read(4))[0]
        
        lengths = {}
        for _ in range(unique_symbols):
            byte_val = f.read(1)[0]
            code_len = f.read(1)[0]
            lengths[byte_val] = code_len
        
        padding = f.read(1)[0]
        total_bits = int.from_bytes(f.read(4), 'big')
        compressed_data = f.read()
        
        encoded_bits = bytes_to_bits(compressed_data, total_bits)

    rle_data = huffman_decode(encoded_bits, lengths)

    mtf_data = deRLE(rle_data, rle_ms, rle_mc)

    bwt_data = mtf_decode(mtf_data)

    if bwt_blocked:
        result = bwt_decode(bwt_data, indices, True)
    else:
        result = bwt_decode(bwt_data, indices[0], False)

    with open(output_path, 'wb') as f:
        f.write(result)
    
TEST_FILES = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/enwik7",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test2.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test3.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test4.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test5.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test6.raw",
]

TEST_FILES_COMPRESS = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/enwik7_comp",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test2_comp.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test3_comp.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test4_comp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test5_comp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test6_comp.raw",
]

TEST_FILES_DECOMPRESS = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/enwik7_decomp",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test2_decomp.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test3_decomp.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test4_decomp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test5_decomp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test6_decomp.raw",
]

from pathlib import Path

for file_path in range(len(TEST_FILES)):
    
    original_size = Path(TEST_FILES[file_path]).stat().st_size
    
    print(f"\nФайл: {file_path+1} ({original_size:,} байт)")

    # Сжатие
    compress(TEST_FILES[file_path], TEST_FILES_COMPRESS[file_path])
    compressed_size = Path(TEST_FILES_COMPRESS[file_path]).stat().st_size
    
    # Распаковка
    decompress(TEST_FILES_COMPRESS[file_path], TEST_FILES_DECOMPRESS[file_path])
    decompressed_size = Path(TEST_FILES_DECOMPRESS[file_path]).stat().st_size
    
    # Проверка целостности
    if decompressed_size == original_size:
        print("correct")
    else:
        print("fail")
    
    # Статистика
    ratio = (original_size / compressed_size) if compressed_size > 0 else 0
    print(f"Исходный размер: {original_size}\n")
    print(f"После сжатия: {compressed_size}\n")
    print(f"После распаковки: {decompressed_size}\n")
    print(f"Коэффициент сжатия: {ratio}\n")


Файл: 1 (10,000,000 байт)
correct
Исходный размер: 10000000

После сжатия: 5319255

После распаковки: 10000000

Коэффициент сжатия: 1.8799625135474798


Файл: 2 (371,816 байт)
correct
Исходный размер: 371816

После сжатия: 117934

После распаковки: 371816

Коэффициент сжатия: 3.152746451405023


Файл: 3 (1,104,440 байт)
correct
Исходный размер: 1104440

После сжатия: 683098

После распаковки: 1104440

Коэффициент сжатия: 1.6168104722894812


Файл: 4 (1,532,562 байт)
correct
Исходный размер: 1532562

После сжатия: 727468

После распаковки: 1532562

Коэффициент сжатия: 2.10670709914388


Файл: 5 (2,001,012 байт)
correct
Исходный размер: 2001012

После сжатия: 1212240

После распаковки: 2001012

Коэффициент сжатия: 1.6506731340328649


Файл: 6 (1,806,348 байт)
correct
Исходный размер: 1806348

После сжатия: 1676314

После распаковки: 1806348

Коэффициент сжатия: 1.0775713857904903

